In [1]:
#%% Imports and settings
import sciris as sc
import starsim as ss
import numpy as np
import pandas as pd
import matplotlib.dates as mdates

n_agents = 100000
debug = False # If true, will run in serial

/Users/nparke19/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class SEIR(ss.SIR):
        def __init__(self, pars=None, *args, **kwargs):
            super().__init__()
            self.define_pars(dur_exp = ss.lognorm_ex(mean = 1.6, std  = 0.3)
            )
            self.update_pars(pars, **kwargs)

            # Additional states beyond the SIR ones
            self.define_states(
                ss.State('exposed', label='Exposed'),
                ss.FloatArr('ti_exposed', label='TIme of exposure'),
            )
            return

        @property
        def infectious(self):
            return self.infected 

        def step_state(self):
            """ Make all the updates from the SIR model """
            # Perform SIR updates
            super().step_state()

            # Additional updates: progress exposed -> infected
            infected = self.exposed & (self.ti_infected <= self.ti)
            self.exposed[infected] = False
            self.infected[infected] = True
            return

        def step_die(self, uids):
            super().step_die(uids)
            self.exposed[uids] = False
            return

        def set_prognoses(self, uids, sources=None):
            """ Carry out state changes associated with infection """
            super().set_prognoses(uids, sources)
            ti = self.ti
            self.susceptible[uids] = False
            self.exposed[uids] = True
            self.ti_exposed[uids] = ti

            # Calculate and schedule future outcomes
            dur_exp = self.pars['dur_exp'].rvs(uids)
            self.ti_infected[uids] = ti + dur_exp
            dur_inf = self.pars['dur_inf'].rvs(uids)
            will_die = self.pars['p_death'].rvs(uids)
            self.ti_recovered[uids[~will_die]] = ti + dur_inf[~will_die]
            self.ti_dead[uids[will_die]] = ti + dur_inf[will_die]
            return

        def plot(self):
            """ Update the plot with the exposed compartment """
            with ss.options.context(jupyter=False):
                fig = super().plot()
                ax = plt.gca()
                res = self.results.n_exposed
                ax.plot(res.timevec, res, label=res.label)
                plt.legend()
            return ss.return_fig(fig)

In [3]:
def make_sim():
    seir = SEIR(
        beta = ss.beta(0.077),
        init_prev = ss.bernoulli(p = 0.0007),
        dur_inf = ss.poisson(7)
    )
    random = ss.RandomNet(n_contacts=ss.poisson(5))

    sim = ss.Sim(
        n_agents = n_agents,
        start = sc.date('2020-9-17'),
        stop = sc.date('2021-03-01'),
        dt = 1,
        unit = 'day',
        diseases = seir,
        networks = random,
        verbose = 0,
    )

    return sim



In [4]:
# Define the calibration parameters
calib_pars = dict(
    beta = dict(low=0.01, high=0.30, guess=0.077, suggest_type='suggest_float')) # Default type is suggest_float, no need to re-specify)

In [5]:
def build_sim(sim, calib_pars, **kwargs):
    """ Modify the base simulation by applying calib_pars """

    reps = kwargs.get('n_reps', 1)

    seir = sim.pars.diseases # There is only one disease in this simulation and it is a SIR


    for k, pars in calib_pars.items():
        if k == 'rand_seed':
            sim.pars.rand_seed = pars
            continue

        v = pars['value']
        if k == 'beta':
            seir.pars.beta = ss.beta(v)
        else:
            raise NotImplementedError(f'Parameter {k} not recognized')

    if reps == 1:
        return sim

    # Ignoring the random seed if provided via the reseed=True option in Calibration
    ms = ss.MultiSim(sim, iterpars=dict(rand_seed=np.random.randint(0, 1e6, reps)), initialize=True, debug=True, parallel=False)
    return ms

In [6]:
def eval(sim, expected):
    """
    Evaluate the simulation against multiple expected (date, value) pairs.
    
    Args:
        sim (Sim or MultiSim): The simulation object to evaluate.
        expected (list of tuples): List of (date, prevalence) pairs to compare against.
        
    Returns:
        float: The total squared error across all dates and sims.
    """
    if not isinstance(sim, ss.MultiSim):
        sim = ss.MultiSim(sims=[sim])

    total_error = 0.0
    for s in sim.sims:
        # Check for extinction (all new infections very low or zero)
        if np.all(s.results.seir.new_infections < 20):
            return 1e8  # Immediately return large penalty

        for date, expected_inc in expected:
            # Find index of the given date in the simulation
            ind = np.searchsorted(s.results.timevec, date, side='left')
            if ind >= len(s.results.timevec):
                ind = -1  # Use last available date if overshooting

            sim_inc = s.results.seir.new_infections[ind]
            total_error += (sim_inc - expected_inc) ** 2

    return total_error

In [7]:
# Convert to list of (ss.date, value) tuples

penn_inc_rates = pd.read_csv("~/Desktop/penn_inc_rates2.csv")

# Convert to list of (ss.date, value) tuples
penn_inc_rates_df = [(ss.date(row['time_value']), row['cases']) for _, row in penn_inc_rates.iterrows()]
penn_inc_rates_df

[(<2020.10.01>, 38.2252325),
 (<2020.10.02>, 38.97398),
 (<2020.10.03>, 40.538527),
 (<2020.10.04>, 40.93525149999999),
 (<2020.10.05>, 41.577833),
 (<2020.10.06>, 39.14161),
 (<2020.10.07>, 41.8516285),
 (<2020.10.08>, 43.2820715),
 (<2020.10.09>, 44.846618500000005),
 (<2020.10.10>, 47.852783499999994),
 (<2020.10.11>, 48.920028),
 (<2020.10.12>, 50.652204999999995),
 (<2020.10.13>, 52.57436249999999),
 (<2020.10.14>, 52.121761500000005),
 (<2020.10.15>, 53.6974835),
 (<2020.10.16>, 54.9602965),
 (<2020.10.17>, 55.83197249999999),
 (<2020.10.18>, 56.446616),
 (<2020.10.19>, 57.1897755),
 (<2020.10.20>, 58.4525885),
 (<2020.10.21>, 59.7321645),
 (<2020.10.22>, 62.509235),
 (<2020.10.23>, 65.4036465),
 (<2020.10.24>, 66.8843785),
 (<2020.10.25>, 69.5049945),
 (<2020.10.26>, 71.93563),
 (<2020.10.27>, 76.5622185),
 (<2020.10.28>, 80.803258),
 (<2020.10.29>, 82.6639515),
 (<2020.10.30>, 84.580521),
 (<2020.10.31>, 87.18437399999999),
 (<2020.11.01>, 89.03948),
 (<2020.11.02>, 90.7046045)

In [9]:

sc.heading('Beginning calibration')

# Make the sim and data
sim = make_sim()

# Make the calibration
calib = ss.Calibration(
    calib_pars = calib_pars,
    sim = sim,
    build_fn = build_sim,
    build_kw = dict(n_reps=5), # Two reps per point
    reseed = True,
    eval_fn = eval, # Will call my_function(msim, eval_kwargs)
    eval_kw = dict(expected=penn_inc_rates_df), # Will call eval(sim, **eval_kw)
    total_trials = 100,
    n_workers = None, # None indicates to use all available CPUs
    die = True,
    debug = debug,
)

# Perform the calibration
sc.printcyan('\nPeforming calibration...')
calib.calibrate()

# Check
calib.check_fit()



—————————————————————
Beginning calibration
—————————————————————


Peforming calibration...
Removed existing calibration file starsim_calibration.db
sqlite:///starsim_calibration.db


[I 2025-08-21 10:01:25,271] A new study created in RDB with name: starsim_calibration


Elapsed time: 35.2 s
Elapsed time: 36.6 s


[I 2025-08-21 10:02:04,521] Trial 11 finished with value: 39383738.37145046 and parameters: {'beta': 0.01040089182642904, 'rand_seed': 646756}. Best is trial 11 with value: 39383738.37145046.
[I 2025-08-21 10:02:04,709] Trial 3 finished with value: 44460553.64074648 and parameters: {'beta': 0.2674864773468983, 'rand_seed': 523910}. Best is trial 11 with value: 39383738.37145046.


Elapsed time: 36.8 s


...

[I 2025-08-21 10:07:39,109] Trial 106 finished with value: 250289134.232797 and parameters: {'beta': 0.04955603966264775, 'rand_seed': 717737}. Best is trial 53 with value: 1531252.0915854648.


Elapsed time: 34.6 s


[I 2025-08-21 10:07:40,419] Trial 107 finished with value: 214301572.34236842 and parameters: {'beta': 0.048769643856031664, 'rand_seed': 954195}. Best is trial 53 with value: 1531252.0915854648.


Making results structure...
Processed 108 trials; 0 failed
Best pars: {'beta': 0.04001725549220916, 'rand_seed': 753532}
Removed existing calibration file starsim_calibration.db

Checking fit...
Elapsed time: 7.17 s
Fit with original pars: 1551877005.933235
Fit with best-fit pars: 1933438.3065184674
✓ Calibration improved fit 1551877005.933235 --> 1933438.3065184674


True

In [10]:
calib.plot();

In [11]:
calib.plot(bootstrap=True); # Pass bootstrap=True to produce this plot
